In [ ]:
%load_ext autoreload
%autoreload 2

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import os
from datasets import load_dataset
import torch
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM, AutoTokenizer
import transformers

root_folder = '/u/eboix/moe_distillation'
if os.path.exists(root_folder):
    os.chdir(root_folder)

from activation_buffer import ActivationDataLoader

# Check for GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


In [5]:
# Choose a Pythia model size (options include: 70m, 160m, 410m, 1b, 1.4b, 2.8b, 6.9b, 12b)
model_name = "EleutherAI/pythia-70m"

# Load the model and tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name).to(device)
tokenizer.pad_token = tokenizer.eos_token

print(f"Loaded {model_name} model with {sum(p.numel() for p in model.parameters())/1e6:.2f}M parameters")

# Load WikiText-2 dataset (smaller than WikiText-103)
dataset = load_dataset("wikitext", "wikitext-2-raw-v1")

# Alternatively, load Wikipedia English dataset
# dataset = load_dataset('wikimedia/wikipedia', '20231101.en', split='train')

print(f"Dataset loaded with {len(dataset)} training examples")


Loaded EleutherAI/pythia-70m model with 70.43M parameters
Dataset loaded with 3 training examples


In [6]:

from activation_buffer import ActivationDataLoader

# Example of how to use this class:
# Process the dataset in batches
batch_size = 1024
layer_idx=3
num_batches_in_val = 100  # Number of batches to collect from validation set

train_text_dataloader = DataLoader(
    dataset['train'], 
    batch_size=1, 
    shuffle=True, 
    collate_fn=lambda x: [item['text'] for item in x]
)

val_text_dataloader = DataLoader(
    dataset['validation'], 
    batch_size=1, 
    shuffle=True, 
    collate_fn=lambda x: [item['text'] for item in x]
)

# Create the activation data loader
train_activation_loader = ActivationDataLoader(
    model=model,
    tokenizer=tokenizer,
    layer_idx=layer_idx,
    max_length=None,
    batch_size=batch_size,
    activation_type='input',
    text_dataloader=train_text_dataloader,
    max_buffer_size=1000000,  # Adjust buffer size as needed
)

# Create the activation data loader
val_activation_loader = ActivationDataLoader(
    model=model,
    tokenizer=tokenizer,
    layer_idx=layer_idx,
    max_length=None,
    batch_size=batch_size,
    activation_type='input',
    text_dataloader=val_text_dataloader,
    max_buffer_size=100000,  # Adjust buffer size as needed
)
train_iterator = iter(train_activation_loader)
val_iterator = iter(val_activation_loader)
# Create a dataset from 100 batches of the validation iterator
val_activations = []
for _ in tqdm(range(num_batches_in_val), desc="Collecting validation activations"):
    try:
        val_activations.append(next(val_iterator))
    except StopIteration:
        break
del val_iterator
del val_activation_loader


# # Concatenate all activations
# all_activations = np.vstack([act.reshape(-1, act.shape[-1]) for act in all_activations])
# print(f"Collected activations shape: {all_activations.shape}")


In [24]:
class ParallelMLPs(nn.Module):
    def __init__(self, input_dim, output_dim, intermediate_dim=128, multi_index_dim=2, m=127, top_k=None, bias=False,
                 init_scale=0.01):
        super(ParallelMLPs, self).__init__()
        self.m = m
        self.intermediate_dim = intermediate_dim
        self.output_dim = output_dim
        self.input_dim = input_dim
        self.multi_index_dim = multi_index_dim
        self.top_k = top_k
        self.bias = bias

        # Parameters E, F, G, H, where
        # E is of dimension (m, output_dim, k)
        self.E = nn.Parameter(torch.randn(m, output_dim, multi_index_dim) * init_scale)
        # F is of dimension (m, k, intermediate_dim)
        self.F = nn.Parameter(torch.randn(m, multi_index_dim, intermediate_dim) * init_scale)
        # G is of dimension (m, intermediate_dim, k)
        self.G = nn.Parameter(torch.randn(m, intermediate_dim, multi_index_dim) * init_scale)
        # H is of dimension (m, k, input_dim)
        self.H = nn.Parameter(torch.randn(m, multi_index_dim, input_dim) * init_scale)
        if bias:
            self.bias_G = nn.Parameter(torch.zeros(m, intermediate_dim,1))
            self.bias_E = nn.Parameter(torch.zeros(output_dim,1))

    def forward(self, x):
        # x is of shape (batch_size, input_dim))
        # left-multiply by H
        x = torch.einsum('mki,bi->mkb', self.H, x)
        # x is now of shape (m, k, batch_size)
        # left-multiply by G
        x = torch.einsum('mtk,mkb->mtb', self.G, x)
        if self.bias:
            # add bias for G
            x = x + self.bias_G
        # x is now of shape (m, intermediate_dim, batch_size)
        # apply GeLU activation elementwise
        x = torch.nn.functional.gelu(x)
        # left-multiply by F
        x = torch.einsum('mkt,mtb->mkb', self.F, x)
        # x is now of shape (m, k, batch_size)
        # left-multiply by E
        x = torch.einsum('mok,mkb->mob', self.E, x)
        # x is now of shape (m, output_dim, batch_size)
        if self.top_k is not None:
            assert(False), "Top-k is not yet supported in this implementation"
        # sum over the first dimension (m)
        x = torch.sum(x, dim=0)
        if self.bias:
            # add bias for E
            x = x + self.bias_E
        # x is now of shape (output_dim, batch_size)
        # transpose to get (batch_size, output_dim)
        x = x.transpose(0, 1)
        return x

In [ ]:
# Teacher model is the MLP of the specified layer
teacher_mlp = model.gpt_neox.layers[layer_idx].mlp
input_dim = teacher_mlp.dense_h_to_4h.in_features
output_dim = input_dim

# Initialize the student model
student_model = ParallelMLPs(
    input_dim=input_dim,
    output_dim=output_dim,
    intermediate_dim=4,
    multi_index_dim=2,  # Number of indices in the multi-index
    m=3200,  # Number of parallel MLPs
    top_k=None,  # Set to None for now, as we are not using top-k
    bias=True  # Bias is not yet supported in this implementation
).to(device)
# Print model summary
print(f"Initialized student model with {sum(p.numel() for p in student_model.parameters())/1e6:.2f}M parameters")


num_train_batches_per_epoch = 100  # Number of batches to train on
num_epochs = 100

# Training parameters
optimizer = optim.AdamW(student_model.parameters(), lr=1e-3)
# Add cosine learning rate scheduler
total_steps = num_train_batches_per_epoch * num_epochs
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_steps)
criterion = nn.MSELoss()

# Compute variance of teacher model output on activation validation set
mean_val_output = torch.zeros(output_dim, device=device)
val_activations_variance = 0.0
for val_activation in val_activations:
    with torch.no_grad():
        teacher_output = teacher_mlp(val_activation)
        mean_val_output += teacher_output.mean(dim=0)
mean_val_output /= len(val_activations)
for val_activation in val_activations:
    with torch.no_grad():
        teacher_output = teacher_mlp(val_activation)
        val_activations_variance += torch.sum((teacher_output - mean_val_output.view(1,-1)) ** 2).item() / teacher_output.shape[0]
val_activations_variance /= len(val_activations)
val_activations_variance /= mean_val_output.shape[0]  # Normalize by output dimension
print(f"Teacher model output variance on validation set: {val_activations_variance:.6f}")

# Training loop
for epoch in range(num_epochs):
    # Training phase
    total_loss = 0.0
    for _ in range(num_train_batches_per_epoch):
        try:
            layer_input = next(train_iterator)
        except StopIteration:
            train_iterator = iter(train_activation_loader)
            layer_input = next(train_iterator)
        with torch.no_grad():
            layer_output = teacher_mlp(layer_input)
        
        student_output = student_model(layer_input)
        loss = criterion(student_output, layer_output)

        # Backward and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()  # Update learning rate
            
        total_loss += loss.item()
    avg_loss = total_loss / num_train_batches_per_epoch
    print(f"Epoch [{epoch+1}/{num_epochs}], Train Loss: {avg_loss:.6f}, LR: {scheduler.get_last_lr()[0]:.6f}")
    
    # Validation phase
    tot_val_loss = 0.0
    for val_activation in val_activations:
        with torch.no_grad():
            teacher_output = teacher_mlp(val_activation)
            student_output = student_model(val_activation)
            val_loss = criterion(student_output, teacher_output).item()
        tot_val_loss += val_loss
    avg_val_loss = tot_val_loss / len(val_activations)
    print(f"Validation Loss: {avg_val_loss:.6f}")
    print(f'Fraction validation loss over variance: {avg_val_loss / val_activations_variance:.6f}')

Initialized student model with 6.62M parameters
Teacher model output variance on validation set: 0.126587
Epoch [1/100], Train Loss: 0.101453, LR: 0.001000
Validation Loss: 0.049071
Fraction validation loss over variance: 0.387651
Epoch [2/100], Train Loss: 0.034749, LR: 0.000999
Validation Loss: 0.026156
Fraction validation loss over variance: 0.206628
Epoch [3/100], Train Loss: 0.022740, LR: 0.000998
Validation Loss: 0.019623
Fraction validation loss over variance: 0.155017
Epoch [4/100], Train Loss: 0.018184, LR: 0.000996
Validation Loss: 0.016711
Fraction validation loss over variance: 0.132013
Epoch [5/100], Train Loss: 0.015984, LR: 0.000994
Validation Loss: 0.014930
Fraction validation loss over variance: 0.117943
Epoch [6/100], Train Loss: 0.014524, LR: 0.000991
Validation Loss: 0.013704
Fraction validation loss over variance: 0.108255
Epoch [7/100], Train Loss: 0.013403, LR: 0.000988
Validation Loss: 0.012777
Fraction validation loss over variance: 0.100936
Epoch [8/100], Trai

In [18]:
type(model.gpt_neox.layers[0].mlp)

transformers.models.gpt_neox.modeling_gpt_neox.GPTNeoXMLP

In [ ]:
# Teacher model is the MLP of the specified layer
teacher_mlp = model.gpt_neox.layers[layer_idx].mlp
input_dim = teacher_mlp.dense_h_to_4h.in_features
output_dim = input_dim

# Initialize the student model
# student_model = nn.Sequential(
#     nn.Linear(input_dim, 2048),
#     nn.GELU(),
#     nn.Linear(2048, output_dim)
# ).to(device)
student_model = nn.Sequential(
    nn.Linear(input_dim, 256),
    nn.GELU(),
    nn.Linear(256, output_dim)
).to(device)

# student_model = transformers.models.gpt_neox.modeling_gpt_neox.GPTNeoXMLP(model.config).to(device)

num_train_batches_per_epoch = 100  # Number of batches to train on
num_epochs = 100

# Training parameters
optimizer = optim.AdamW(student_model.parameters(), lr=3e-4)
# Add cosine learning rate scheduler
total_steps = num_train_batches_per_epoch * num_epochs
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_steps)
criterion = nn.MSELoss()

# Compute variance of teacher model output on activation validation set
mean_val_output = torch.zeros(output_dim, device=device)
val_activations_variance = 0.0
for val_activation in val_activations:
    with torch.no_grad():
        teacher_output = teacher_mlp(val_activation)
        mean_val_output += teacher_output.mean(dim=0)
mean_val_output /= len(val_activations)
for val_activation in val_activations:
    with torch.no_grad():
        teacher_output = teacher_mlp(val_activation)
        val_activations_variance += torch.sum((teacher_output - mean_val_output.view(1,-1)) ** 2).item() / teacher_output.shape[0]
val_activations_variance /= len(val_activations)
val_activations_variance /= mean_val_output.shape[0]  # Normalize by output dimension
print(f"Teacher model output variance on validation set: {val_activations_variance:.6f}")

# Training loop
for epoch in range(num_epochs):
    # Training phase
    total_loss = 0.0
    for _ in range(num_train_batches_per_epoch):
        try:
            layer_input = next(train_iterator)
        except StopIteration:
            train_iterator = iter(train_activation_loader)
            layer_input = next(train_iterator)
        with torch.no_grad():
            layer_output = teacher_mlp(layer_input)
        
        student_output = student_model(layer_input)
        loss = criterion(student_output, layer_output)

        # Backward and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()  # Update learning rate
            
        total_loss += loss.item()
    avg_loss = total_loss / num_train_batches_per_epoch
    print(f"Epoch [{epoch+1}/{num_epochs}], Train Loss: {avg_loss:.6f}, LR: {scheduler.get_last_lr()[0]:.6f}")
    
    # Validation phase
    tot_val_loss = 0.0
    for val_activation in val_activations:
        with torch.no_grad():
            teacher_output = teacher_mlp(val_activation)
            student_output = student_model(val_activation)
            val_loss = criterion(student_output, teacher_output).item()
        tot_val_loss += val_loss
    avg_val_loss = tot_val_loss / len(val_activations)
    print(f"Validation Loss: {avg_val_loss:.6f}")
    print(f'Fraction validation loss over variance: {avg_val_loss / val_activations_variance:.6f}')

Teacher model output variance on validation set: 0.126696


Epoch [1/100], Train Loss: 0.045327, LR: 0.000300
Validation Loss: 0.026770
Fraction validation loss over variance: 0.211297
Epoch [2/100], Train Loss: 0.024751, LR: 0.000300
Validation Loss: 0.023106
Fraction validation loss over variance: 0.182373
Epoch [3/100], Train Loss: 0.022680, LR: 0.000299
Validation Loss: 0.021893
Fraction validation loss over variance: 0.172803


KeyboardInterrupt: 

# MoE training

In [22]:
# Define MLP class for MoE experts
class MLP(nn.Module):
    def __init__(self, input_dim, output_dim, hidden_dim=32):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, output_dim)
        )

    def forward(self, x):
        return self.fc(x)

# Define MoE with Top-K (where each expert is an MLP)
class MoE_TopK(nn.Module):
    def __init__(self, input_dim, output_dim, num_experts, k, hidden_dim=32,bias=False):
        super().__init__()
        self.num_experts = num_experts
        self.k = k  # Number of selected experts

        # Gating network
        self.gate = nn.Linear(input_dim, num_experts,bias=bias)

        self.expert_tally = nn.Parameter(torch.zeros(self.num_experts))
        self.expert_tally.requires_grad = False

        # Expert networks (each expert is an MLP)
        self.experts = nn.ModuleList([MLP(input_dim, output_dim, hidden_dim) for _ in range(num_experts)])

    def forward(self, x):
        gate_scores = self.gate(x)  # (batch_size, num_experts) 
        topk_vals, topk_idxs = torch.topk(gate_scores, self.k, dim=-1)  # Get top-k expert indices
        # print(topk_idxs.shape)
        # Count the number of occurrences of each index in topk_idxs
        # unique_idxs, counts = torch.unique(topk_idxs, return_counts=True)
        # print(unique_idxs)
        # print(counts)
        # print(self.expert_tally)
        # self.expert_tally[unique_idxs] += counts
        # print('before',self.expert_tally)
        # print(topk_idxs.shape)
        # for i in range(topk_idxs.shape[0]):
        #     for j in range(topk_idxs.shape[1]):
        #         print(i,j)
        #         self.expert_tally[topk_idxs[j]] += 1
        # print('after',self.expert_tally)
        topk_weights = torch.softmax(topk_vals, dim=-1)  # Normalize weights over top-k

        batch_size, _ = x.shape
        expert_outputs = torch.stack([self.experts[i](x) for i in range(self.num_experts)], dim=1)  # (batch_size, num_experts, output_dim)
        selected_expert_outputs = torch.gather(expert_outputs, 1, topk_idxs.unsqueeze(-1).expand(-1, -1, expert_outputs.shape[-1]))

        output = torch.sum(selected_expert_outputs * topk_weights.unsqueeze(-1), dim=1)
        return output

In [ ]:
# Teacher model is the MLP of the specified layer
teacher_mlp = model.gpt_neox.layers[layer_idx].mlp
input_dim = teacher_mlp.dense_h_to_4h.in_features
output_dim = input_dim

# Initialize the student model as a MoE model
num_experts = 256
k = 8  # Number of active experts
hidden_dim = 32  # Smaller hidden dimension for each expert

# Create MoE student model
student_model = MoE_TopK(
    input_dim=input_dim, 
    output_dim=output_dim, 
    num_experts=num_experts, 
    k=k,
    hidden_dim=hidden_dim
).to(device)

num_train_batches_per_epoch = 1000  # Number of batches to train on
num_epochs = 100

# Training parameters
optimizer = optim.AdamW(student_model.parameters(), lr=3e-3)
# Add cosine learning rate scheduler
total_steps = num_train_batches_per_epoch * num_epochs
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_steps)
criterion = nn.MSELoss()

# Compute variance of teacher model output on activation validation set
mean_val_output = torch.zeros(output_dim, device=device)
val_activations_variance = 0.0
for val_activation in val_activations:
    with torch.no_grad():
        teacher_output = teacher_mlp(val_activation)
        mean_val_output += teacher_output.mean(dim=0)
mean_val_output /= len(val_activations)
for val_activation in val_activations:
    with torch.no_grad():
        teacher_output = teacher_mlp(val_activation)
        val_activations_variance += torch.sum((teacher_output - mean_val_output.view(1,-1)) ** 2).item() / teacher_output.shape[0]
val_activations_variance /= len(val_activations)
val_activations_variance /= mean_val_output.shape[0]  # Normalize by output dimension
print(f"Teacher model output variance on validation set: {val_activations_variance:.6f}")

# Training loop
for epoch in range(num_epochs):
    # Training phase
    total_loss = 0.0
    for _ in range(num_train_batches_per_epoch):
        try:
            layer_input = next(train_iterator)
        except StopIteration:
            train_iterator = iter(train_activation_loader)
            layer_input = next(train_iterator)
        with torch.no_grad():
            layer_output = teacher_mlp(layer_input)
        
        student_output = student_model(layer_input)
        loss = criterion(student_output, layer_output)

        # Backward and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()  # Update learning rate
            
        total_loss += loss.item()
    avg_loss = total_loss / num_train_batches_per_epoch
    print(f"Epoch [{epoch+1}/{num_epochs}], Train Loss: {avg_loss:.6f}, LR: {scheduler.get_last_lr()[0]:.6f}")
    
    # Validation phase
    tot_val_loss = 0.0
    for val_activation in val_activations:
        with torch.no_grad():
            teacher_output = teacher_mlp(val_activation)
            student_output = student_model(val_activation)
            val_loss = criterion(student_output, teacher_output).item()
        tot_val_loss += val_loss
    avg_val_loss = tot_val_loss / len(val_activations)
    print(f"Validation Loss: {avg_val_loss:.6f}")
    print(f'Fraction validation loss over variance: {avg_val_loss / val_activations_variance:.6f}')

Teacher model output variance on validation set: 0.126696
Epoch [1/100], Train Loss: 0.033964, LR: 0.002999
Validation Loss: 0.030159
Fraction validation loss over variance: 0.238042


KeyboardInterrupt: 